<a href="https://colab.research.google.com/github/CAWARI0208/LLMs-from-scratch/blob/main/dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 徹底清空目前的環境（確保沒有殘留的髒檔案）
%cd /content
!rm -rf LLMs-from-scratch

# 2. 複製你 Fork 後的完整 GitHub 專案
# （請把下面的「你的帳號」換成你自己的 GitHub 帳號名稱）
!git clone https://github.com/CAWARI0208/LLMs-from-scratch

# 3. 將 Colab 的工作目錄「切換進去」專案資料夾內（非常關鍵！）
%cd /content/LLMs-from-scratch

# 4. 安裝專案執行所需的 Python 附件套件
!pip install tiktoken tqdm matplotlib

# 5. 檢查當前目錄，確認附件都在
!ls -R


/content
Cloning into 'LLMs-from-scratch'...
remote: Enumerating objects: 7348, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 7348 (delta 5), reused 1 (delta 1), pack-reused 7336 (from 2)
Receiving objects: 100% (7348/7348), 16.52 MiB | 15.03 MiB/s, done.
Resolving deltas: 100% (4486/4486), done.
/content/LLMs-from-scratch
.:
appendix-A  ch01  ch06		pixi.toml		requirements.txt
appendix-B  ch02  ch07		pkg			setup
appendix-C  ch03  CITATION.cff	pyproject.toml		troubleshooting.md
appendix-D  ch04  conftest.py	README.md
appendix-E  ch05  LICENSE.txt	reasoning-from-scratch

./appendix-A:
01_main-chapter-code  02_setup-recommendations	README.md

./appendix-A/01_main-chapter-code:
code-part1.ipynb  DDP-script.py		  exercise-solutions.ipynb
code-part2.ipynb  DDP-script-torchrun.py  README.md

./appendix-A/02_setup-recommendations:
README.md

./appendix-B:
README.md

./appendix-C:
README.md

./appendix-D:
01_main-chapter-code 

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# The Main Data Loading Pipeline Summarized

The complete chapter code is located in [ch02.ipynb](./ch02.ipynb).

This notebook contains the main takeaway, the data loading pipeline without the intermediate steps.

Packages that are being used in this notebook:

In [ ]:
# NBVAL_SKIP
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0+cpu
tiktoken version: 0.13.0


In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


with open("/content/LLMs-from-scratch/ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

vocab_size = 50257
output_dim = 256
context_length = 1024


token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

batch_size = 8
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

In [ ]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [ ]:
print(input_embeddings.shape)

torch.Size([8, 4, 256])
